# AI-Powered E-Commerce Product Intelligence System
### Exploratory Notebook

This notebook walks through the core ideas behind the project. For production use, see `src/` and `app.py`.

**Features covered here:**
1. Dataset exploration (styles.csv)
2. CLIP embedding generation
3. FAISS index construction
4. Text-to-product search demo
5. Image-to-product search demo
6. Complementary recommendations demo
7. Duplicate detection demo
8. Zero-shot AI classification demo

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '../')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

print('Libraries loaded ✅')

## 1. Dataset Exploration

In [ ]:
from src.utils import load_metadata

df = load_metadata('../data/styles.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Distribution of top-level categories
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['gender', 'masterCategory', 'subCategory']):
    vc = df[col].value_counts().head(10)
    vc.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Top {col} values', fontsize=13)
    ax.set_xlabel('Count')

plt.tight_layout()
plt.savefig('../screenshots/dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to screenshots/ ✅')

## 2. CLIP Embedding Generation

In [ ]:
from src.embeddings import load_clip_model, get_image_embedding, get_text_embedding

model, preprocess = load_clip_model('ViT-B/32')
print('CLIP model loaded ✅')

# Demo: text embedding
text_emb = get_text_embedding('blue casual shirt', model)
print(f'Text embedding shape: {text_emb.shape}')   # (512,)
print(f'L2 norm: {np.linalg.norm(text_emb):.4f}')  # should be ~1.0

## 3. Text Search Demo

In [ ]:
from src.embeddings import load_embeddings
from src.search import load_faiss_index, search_by_text

# Load pre-built index (run build_index.py first)
embeddings, image_paths = load_embeddings('../data/')
faiss_index = load_faiss_index('../data/faiss.index')

query = 'black running shoes for men'
results = search_by_text(query, model, faiss_index, image_paths, df, top_k=8)
print(results[['rank', 'score', 'productDisplayName', 'subCategory']].to_string(index=False))

In [ ]:
# Visualise text search results
def show_results(result_df, title='Search Results', cols=4):
    n = len(result_df)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 4))
    axes = axes.flatten() if n > 1 else [axes]
    
    for i, (_, row) in enumerate(result_df.iterrows()):
        try:
            img = mpimg.imread(row['image_path'])
            axes[i].imshow(img)
        except Exception:
            axes[i].text(0.5, 0.5, 'No image', ha='center', va='center')
        axes[i].set_title(f"Score: {row.get('score', 0):.3f}\n{str(row.get('productDisplayName',''))[:30]}",
                          fontsize=8)
        axes[i].axis('off')
    
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'../screenshots/{title.lower().replace(" ", "_")}.png', dpi=120)
    plt.show()

show_results(results, title='Text Search: black running shoes')

## 4. Duplicate Detection Demo

In [ ]:
from src.duplicate_detection import find_duplicates, duplicate_statistics

# Use a subset for demo speed
subset_embs = embeddings[:2000]
subset_paths = image_paths[:2000]

groups = find_duplicates(subset_embs, subset_paths, df, threshold=0.97)
stats = duplicate_statistics(groups, len(subset_paths))

print('\n--- Duplicate Statistics ---')
for k, v in stats.items():
    print(f'  {k}: {v}')

## 5. AI Classification Demo

In [ ]:
from src.classification import clip_zero_shot_classify, format_prediction

# classify a sample image
sample_image = image_paths[42]   # change index to try different products

pred = clip_zero_shot_classify(sample_image, model, preprocess)
print(f'Image: {os.path.basename(sample_image)}')
print(f'Prediction: {format_prediction(pred)}')
print(f'Confidence: {pred["confidence"]}')

# show image
img = Image.open(sample_image)
plt.imshow(img)
plt.title(format_prediction(pred), fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()